# Hallmark Demo: add, commit, checkout, and status

This notebook shows the current Python and CLI workflow:

1. Initialize a hallmark repo
2. Create files and add them with one branch fmt
3. Commit and inspect `config.yml` and `data.tsv`
4. Use `repo.add(".")` to rebuild a branch manifest
5. Checkout between branches and restore files
6. See how staged changes block checkout


## Setup


Auto-reload modules when they change.


In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
%%bash
rm -rf repo1
rm -rf repo2.hm
mkdir repo1


## 1. Initialize the hallmark repo

A fresh repo starts with a commented `config.yml` template. Use it only if your branch fmt needs regex substitutions.


In [3]:
from hallmark import Repo

repo = Repo.init("repo1")


In [4]:
%%bash
echo "=== .hm directory ==="
ls repo1/.hm/
echo ""
echo "=== initial config.yml ==="
cat repo1/.hm/config.yml
echo ""
echo "=== objects stored so far ==="
find repo1/.hm/objects -type f 2>/dev/null | wc -l


=== .hm directory ===
config.yml
data.tsv
meta.yml
README.md

=== initial config.yml ===
# Edit this file only if your branch needs regex substitutions or a preset remote.
# For simple names, you can just run: hallmark add "a{a}_i{i}.h5"
data:
  -
    # fmt: "{release}_{source}_{year}_{doy:03d}_{band}.uvfits"
    encoding:
      # aspin: m([0-9]+(\.[0-9]+)?|\.[0-9]+)
remote:
  # name: origin
  # url: https://example.com/path/to/data/

=== objects stored so far ===
       0


## 2. Create data files on `main`


In [5]:
%%bash
cd repo1
for f in a{0,0.75}_i{0,30,60,90,120}.h5; do
    echo "$f" > "$f"
done
echo "Files in repo1/:"
ls *.h5


Files in repo1/:
a0_i0.h5
a0_i120.h5
a0_i30.h5
a0_i60.h5
a0_i90.h5
a0.75_i0.h5
a0.75_i120.h5
a0.75_i30.h5
a0.75_i60.h5
a0.75_i90.h5


## 3. Add files and commit on `main`

`repo.add("a{a}_i{i}.h5")` sets the branch fmt and builds a manifest with `sha1`, `a`, and `i`.


In [6]:
pf = repo.add("a{a}_i{i}.h5")
print(pf)


            path     a      i
0    a0.75_i0.h5  0.75    0.0
1  a0.75_i120.h5  0.75  120.0
2   a0.75_i30.h5  0.75   30.0
3   a0.75_i60.h5  0.75   60.0
4   a0.75_i90.h5  0.75   90.0
5       a0_i0.h5  0.00    0.0
6     a0_i120.h5  0.00  120.0
7      a0_i30.h5  0.00   30.0
8      a0_i60.h5  0.00   60.0
9      a0_i90.h5  0.00   90.0


In [7]:
%%bash
echo "=== config.yml after add ==="
cat repo1/.hm/config.yml
echo ""
echo "=== data.tsv after add ==="
cat repo1/.hm/data.tsv


=== config.yml after add ===
data:
- fmt: a{a}_i{i}.h5
  encoding: null
remote: null

=== data.tsv after add ===
sha1	a	i
f656d419104e7af783c119186a4d81b15563310f	0.75	0
ec4dae9f0b8f69ac42f0c290fc84e0d6f1bec6cd	0.75	120
890b2ea509575be9bc74ad515731a2562cd7406e	0.75	30
d9cfcdd0b44d81c10f88e126d5b1eea4914259a0	0.75	60
ed2cfc6bb157a42729d3c6f45b228b432edec4be	0.75	90
18682593a012503b71935017dbc044665254a23c	0	0
1c057a4885fd04379ee1217375720665713c902f	0	120
829b87d845b367a9cc87df6ff510d1bdb7444aca	0	30
075a30e0a9c4dddd92447f463219d4c2a842883d	0	60
de40f4983d86342b03809bfcc09958f43f359b89	0	90


In [8]:
repo.commit("add main dataset")


True

In [9]:
%%bash
echo "Objects stored after commit:"
find repo1/.hm/objects -type f | wc -l


Objects stored after commit:
      10


### Clean status


In [10]:
%%bash
cd repo1
hallmark status


On branch main

Changes to be committed:

Changes not staged for commit:

nothing to commit, working tree clean


### Current add modes

- `repo.add("a{a}_i{i}.h5")` uses a format string and can still set the branch fmt
- `repo.set_config(fmt="b{a}_i{i}.h5")` is the explicit way to change branch config
- `repo.add(".")` rebuilds the manifest from current files that match the branch fmt
- explicit file-list adds such as `hallmark add *` are not supported yet with the parameter-based manifest


## 4. Demo: `repo.add(".")` removes deleted files from the manifest

We use a temporary branch so the main workflow stays intact.
On this branch we switch to `b*` files first so it is easy to distinguish from `main`.
Because the filename family changes, we update the branch fmt explicitly with `repo.set_config(...)` before using `repo.add(".")`.


In [11]:
repo.checkout("sync-demo")


True

In [12]:
%%bash
cd repo1
rm a*.h5
for f in b{0,0.75}_i{0,30,60,90,120}.h5; do
    echo "$f" > "$f"
done
echo "Files after switching sync-demo to b files:"
ls *.h5


Files after switching sync-demo to b files:
b0_i0.h5
b0_i120.h5
b0_i30.h5
b0_i60.h5
b0_i90.h5
b0.75_i0.h5
b0.75_i120.h5
b0.75_i30.h5
b0.75_i60.h5
b0.75_i90.h5


In [13]:
repo.set_config(fmt="b{a}_i{i}.h5")


{'data': [{'fmt': 'b{a}_i{i}.h5', 'encoding': None}], 'remote': None}

In [14]:
%%bash
cd repo1
echo "config.yml after repo.set_config:"
cat .hm/config.yml


config.yml after repo.set_config:
data:
- fmt: b{a}_i{i}.h5
  encoding: null
remote: null


In [15]:
%%bash
cd repo1
rm b0.75_i{0,30,60,90,120}.h5
echo "Files after deleting some tracked b files:"
ls *.h5


Files after deleting some tracked b files:
b0_i0.h5
b0_i120.h5
b0_i30.h5
b0_i60.h5
b0_i90.h5


In [16]:
repo.add(".")


,path,a,i
0,b0_i0.h5,0.0,0.0
1,b0_i120.h5,0.0,120.0
2,b0_i30.h5,0.0,30.0
3,b0_i60.h5,0.0,60.0
4,b0_i90.h5,0.0,90.0


In [17]:
%%bash
cd repo1
echo "Manifest after repo.add dot:"
cat .hm/data.tsv


Manifest after repo.add dot:
sha1	a	i
56676e3e5629923617f8c9e542bd1c2ccaf8fa3a	0	0
683e82f624b175c075ac3ea0cca56e94cd7422f8	0	120
cfdbae05e3afc29354d948a32263f7ac1ed5850b	0	30
3a947fb1280b5bca62615d6c7543b88080588e7a	0	60
1711d3b47eac6ae479b6f0f240f83f41b393ebb4	0	90


In [18]:
%%bash
cd repo1
hallmark status


On branch sync-demo

Changes to be committed:
  new file:   b0_i0.h5
  new file:   b0_i120.h5
  new file:   b0_i30.h5
  new file:   b0_i60.h5
  new file:   b0_i90.h5
  deleted:   a0.75_i0.h5
  deleted:   a0.75_i120.h5
  deleted:   a0.75_i30.h5
  deleted:   a0.75_i60.h5
  deleted:   a0.75_i90.h5
  deleted:   a0_i0.h5
  deleted:   a0_i120.h5
  deleted:   a0_i30.h5
  deleted:   a0_i60.h5
  deleted:   a0_i90.h5

Changes not staged for commit:


In [19]:
repo.commit("sync manifest with repo.add dot")


True

In [20]:
repo.checkout("main")


True

In [21]:
%%bash
echo "Files after checkout back to main:"
ls repo1/*.h5


Files after checkout back to main:
repo1/a0_i0.h5
repo1/a0_i120.h5
repo1/a0_i30.h5
repo1/a0_i60.h5
repo1/a0_i90.h5
repo1/a0.75_i0.h5
repo1/a0.75_i120.h5
repo1/a0.75_i30.h5
repo1/a0.75_i60.h5
repo1/a0.75_i90.h5


## 5. Checkout a new branch

The new branch inherits the same fmt from `main`.


In [22]:
repo.checkout("experiment")


True

In [23]:
%%bash
echo "Current .hm branch:"
git -C repo1/.hm branch --show-current
echo ""
echo "config.yml on experiment:"
cat repo1/.hm/config.yml
echo ""
echo "Files after checkout:"
ls repo1/*.h5


Current .hm branch:
experiment

config.yml on experiment:
data:
- fmt: a{a}_i{i}.h5
  encoding: null
remote: null

Files after checkout:
repo1/a0_i0.h5
repo1/a0_i120.h5
repo1/a0_i30.h5
repo1/a0_i60.h5
repo1/a0_i90.h5
repo1/a0.75_i0.h5
repo1/a0.75_i120.h5
repo1/a0.75_i30.h5
repo1/a0.75_i60.h5
repo1/a0.75_i90.h5


## 6. Replace the experiment branch dataset with new files that still match the same fmt


In [24]:
%%bash
cd repo1
rm a*.h5
for f in a{1,1.5}_i{15,45,75,105,110}.h5; do
    echo "$f" > "$f"
done
echo "Files after replacing the dataset on experiment:"
ls *.h5


Files after replacing the dataset on experiment:
a1_i105.h5
a1_i110.h5
a1_i15.h5
a1_i45.h5
a1_i75.h5
a1.5_i105.h5
a1.5_i110.h5
a1.5_i15.h5
a1.5_i45.h5
a1.5_i75.h5


In [25]:
pf_exp = repo.add(".")

print(pf_exp)



           path    a      i
0  a1.5_i105.h5  1.5  105.0
1  a1.5_i110.h5  1.5  110.0
2   a1.5_i15.h5  1.5   15.0
3   a1.5_i45.h5  1.5   45.0
4   a1.5_i75.h5  1.5   75.0
5    a1_i105.h5  1.0  105.0
6    a1_i110.h5  1.0  110.0
7     a1_i15.h5  1.0   15.0
8     a1_i45.h5  1.0   45.0
9     a1_i75.h5  1.0   75.0


In [26]:
%%bash
cd repo1
hallmark status

On branch experiment

Changes to be committed:
  new file:   a1.5_i105.h5
  new file:   a1.5_i110.h5
  new file:   a1.5_i15.h5
  new file:   a1.5_i45.h5
  new file:   a1.5_i75.h5
  new file:   a1_i105.h5
  new file:   a1_i110.h5
  new file:   a1_i15.h5
  new file:   a1_i45.h5
  new file:   a1_i75.h5
  deleted:   a0.75_i0.h5
  deleted:   a0.75_i120.h5
  deleted:   a0.75_i30.h5
  deleted:   a0.75_i60.h5
  deleted:   a0.75_i90.h5
  deleted:   a0_i0.h5
  deleted:   a0_i120.h5
  deleted:   a0_i30.h5
  deleted:   a0_i60.h5
  deleted:   a0_i90.h5

Changes not staged for commit:


In [27]:
%%bash
echo "experiment data.tsv before commit:"
cat repo1/.hm/data.tsv


experiment data.tsv before commit:
sha1	a	i
f0ca3ed468c8796645672d2ebb48d48ae49b6d3c	1.5	105
9d1801d46e071b98a87afcd1edc59fe9302b9a4b	1.5	110
ed34eda23a05ea4c3cd1bf41988c6e2f76cf41e1	1.5	15
b47e5f555daf30afea5b0f99cadad1e2d8c689ec	1.5	45
b2a40517fd5681088f88903165ae54af3c75cf25	1.5	75
635b4b44a00c0a2bde5a7f0e263bb98dcbdae6e2	1	105
24ef5885eff61f1a8fcd77c19329a2651eca0228	1	110
577954873e813a982f70867b49a374a230df3650	1	15
1580545be2a5d485338e3538c7b16709d5fa6e4a	1	45
e2406e4872637ef09f574731be5b5ba5e8aa5761	1	75


In [28]:
repo.commit("replace experiment dataset")


True

In [29]:
%%bash
echo "Objects stored after experiment commit:"
find repo1/.hm/objects -type f | wc -l


Objects stored after experiment commit:
      25


## 7. Checkout back to `main`


In [30]:
repo.checkout("main")


True

In [31]:
%%bash
echo "Files after checkout main:"
ls repo1/*.h5
echo ""
echo "main data.tsv:"
cat repo1/.hm/data.tsv


Files after checkout main:
repo1/a0_i0.h5
repo1/a0_i120.h5
repo1/a0_i30.h5
repo1/a0_i60.h5
repo1/a0_i90.h5
repo1/a0.75_i0.h5
repo1/a0.75_i120.h5
repo1/a0.75_i30.h5
repo1/a0.75_i60.h5
repo1/a0.75_i90.h5

main data.tsv:
sha1	a	i
f656d419104e7af783c119186a4d81b15563310f	0.75	0
ec4dae9f0b8f69ac42f0c290fc84e0d6f1bec6cd	0.75	120
890b2ea509575be9bc74ad515731a2562cd7406e	0.75	30
d9cfcdd0b44d81c10f88e126d5b1eea4914259a0	0.75	60
ed2cfc6bb157a42729d3c6f45b228b432edec4be	0.75	90
18682593a012503b71935017dbc044665254a23c	0	0
1c057a4885fd04379ee1217375720665713c902f	0	120
829b87d845b367a9cc87df6ff510d1bdb7444aca	0	30
075a30e0a9c4dddd92447f463219d4c2a842883d	0	60
de40f4983d86342b03809bfcc09958f43f359b89	0	90


## 8. Checkout back to `experiment` to verify


In [32]:
repo.checkout("experiment")


True

In [33]:
%%bash
echo "Files after checkout experiment:"
ls repo1/*.h5
echo ""
echo "experiment data.tsv:"
cat repo1/.hm/data.tsv


Files after checkout experiment:
repo1/a1_i105.h5
repo1/a1_i110.h5
repo1/a1_i15.h5
repo1/a1_i45.h5
repo1/a1_i75.h5
repo1/a1.5_i105.h5
repo1/a1.5_i110.h5
repo1/a1.5_i15.h5
repo1/a1.5_i45.h5
repo1/a1.5_i75.h5

experiment data.tsv:
sha1	a	i
f0ca3ed468c8796645672d2ebb48d48ae49b6d3c	1.5	105
9d1801d46e071b98a87afcd1edc59fe9302b9a4b	1.5	110
ed34eda23a05ea4c3cd1bf41988c6e2f76cf41e1	1.5	15
b47e5f555daf30afea5b0f99cadad1e2d8c689ec	1.5	45
b2a40517fd5681088f88903165ae54af3c75cf25	1.5	75
635b4b44a00c0a2bde5a7f0e263bb98dcbdae6e2	1	105
24ef5885eff61f1a8fcd77c19329a2651eca0228	1	110
577954873e813a982f70867b49a374a230df3650	1	15
1580545be2a5d485338e3538c7b16709d5fa6e4a	1	45
e2406e4872637ef09f574731be5b5ba5e8aa5761	1	75


## 9. Uncommitted changes block checkout


In [34]:
%%bash
cd repo1
echo "a2_i15.h5" > a2_i15.h5
echo "Files after creating a2_i15.h5:"
ls *.h5


Files after creating a2_i15.h5:
a1_i105.h5
a1_i110.h5
a1_i15.h5
a1_i45.h5
a1_i75.h5
a1.5_i105.h5
a1.5_i110.h5
a1.5_i15.h5
a1.5_i45.h5
a1.5_i75.h5
a2_i15.h5


In [35]:
repo.add(".")


,path,a,i
0,a1.5_i105.h5,1.5,105.0
1,a1.5_i110.h5,1.5,110.0
2,a1.5_i15.h5,1.5,15.0
3,a1.5_i45.h5,1.5,45.0
4,a1.5_i75.h5,1.5,75.0
5,a1_i105.h5,1.0,105.0
6,a1_i110.h5,1.0,110.0
7,a1_i15.h5,1.0,15.0
8,a1_i45.h5,1.0,45.0
9,a1_i75.h5,1.0,75.0


### Dirty status


In [36]:
%%bash
cd repo1
hallmark status


On branch experiment

Changes to be committed:
  new file:   a2_i15.h5

Changes not staged for commit:


In [37]:
try:
    repo.checkout("main")
except RuntimeError as e:
    print(f"Error: {e}")


Error: you have uncommitted hallmark state changes — commit them before checkout


To fix this, commit the staged changes first or discard them.
